# SDR Swap Monitor — EUR Vanilla IRS

Live table of vanilla interest-rate swap trades disseminated through the Swap
Data Repository (`SDR <GO>` → asset class **Rates**, tab **Vanilla**).

**Filters applied**
- Currency = **EUR** (switchable in the UI)
- Platform **excludes** `TWSF`, `TREU`, `BBSF` (Tradeweb SEF / Tradeweb EU / Bloomberg SEF)
- Vanilla fixed-float IRS only (swaptions, caps/floors, FRAs, basis, XCCY, inflation dropped)

**Output columns:** `tenor · rate · notional · dv01 · index · time · platform · related`
- **tenor** — expiration − effective date on American-format dates (`MM/DD/YYYY`),
  snapped to the standard grid with tolerance for business-day adjustment
  (e.g. 11/08/2026 → 11/10/2036 still labels **10y**); genuinely broken dates show as decimals (`9.71y`)
- **dv01** — par-annuity approximation `notional × (1−(1+r)^−T)/r × 1bp`,
  discounted for forward starts
- **time** — execution timestamp, sorted **newest first**
- **related** — trades submitted at the *same* timestamp are tagged as a
  package (`P1`, `P2`, …); they are legs of one risk transfer (curve trades,
  butterflies, compression)

**How to run:** upload this notebook to BQuant and *Run All Cells*. The app is
the last cell. It refreshes automatically at the configured interval.

**Data source** — pick in the UI:
1. **SDR feed (BQL)** — paste your account's SDR query string into
   `SDR_BQL_QUERY` in the config cell (SDR dissemination data is an
   entitlement-dependent dataset; grab the query from your BQuant data
   catalog or Bloomberg rep). Columns are auto-mapped, so any reasonable
   schema works.
2. **CSV export** — from `SDR <GO>` Vanilla tab: *Actions → Export*, save as
   `sdr_export.csv` next to this notebook. The monitor re-reads it every
   refresh, so a periodically re-exported/dropped file behaves like a tick.
3. **Demo data** — synthetic trades so the pipeline and UI can be exercised
   immediately, clearly labelled.

> ⚠️ BQL only runs inside BQuant — this notebook is untested outside the
> Terminal. The data layer is isolated in one cell so any feed issue is easy
> to localise.


In [ ]:
# ---------------- config: all tunables live here ----------------

CCY_DEFAULT       = "EUR"                       # currency filter
CCY_OPTIONS       = ["EUR", "USD", "GBP", "JPY", "CHF"]
EXCLUDE_PLATFORMS = ("TWSF", "TREU", "BBSF")    # venues to ignore
REFRESH_SECONDS   = 30                          # auto-refresh interval
MAX_ROWS          = 500                         # cap rows shown in the grid

# CSV export from SDR <GO> (Rates / Vanilla tab), saved next to the notebook
CSV_PATH = "sdr_export.csv"

# String-form BQL query for the SDR dissemination dataset, if your account is
# entitled to it. Leave empty to disable the BQL source. Example shape:
# SDR_BQL_QUERY = """
#     get(trade_time, effective_date, expiration_date, currency, fixed_rate,
#         notional_amount, floating_rate_index, execution_venue, product)
#     for(sdruniv(asset_class='Rates'))
# """
SDR_BQL_QUERY = ""


In [ ]:
# ---------------- environment ----------------
import threading
from datetime import datetime

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

try:
    import bql
    bq = bql.Service()
except Exception as _e:          # outside BQuant, or service hiccup
    bql, bq = None, None
    print(f"bql unavailable ({_e}) - BQL source disabled, CSV/Demo still work")

try:
    from ipydatagrid import DataGrid
    HAS_GRID = True
except Exception:
    HAS_GRID = False             # falls back to a plain HTML table

try:
    from zoneinfo import ZoneInfo
    APP_TZ = ZoneInfo("Europe/London")
except Exception:
    APP_TZ = None


In [ ]:
# ---------------- core parsing / compute logic ----------------
import numpy as np
import pandas as pd

# ---------------- tenor ----------------
# Standard tenor grid: label -> years. Business-day adjustment (modified
# following) and T+2 effective dates mean the raw day count is rarely an exact
# round number, so we snap to the nearest grid point within a tolerance that
# scales with maturity.
_TENOR_GRID = (
    [(f"{m}m", m / 12.0) for m in (1, 2, 3, 4, 6, 9, 18)]
    + [(f"{y}y", float(y)) for y in list(range(1, 16)) + [20, 25, 30, 35, 40, 45, 50]]
)


def year_frac(effective, maturity):
    """ACT/365.25 year fraction between two timestamps."""
    return (maturity - effective).days / 365.25


def snap_tenor(yf):
    """Snap a raw year fraction to the standard tenor grid.

    Tolerance = max(18 calendar days, 1.5% of the tenor) so that holiday /
    modified-following adjusted dates (e.g. 10y printed as 10y + 3 business
    days) still label as the standard tenor, while genuinely broken dates
    fall through to a decimal label like '9.71y'.
    """
    if not np.isfinite(yf) or yf <= 0:
        return None
    label, gy = min(_TENOR_GRID, key=lambda t: abs(yf - t[1]))
    tol = max(0.05, 0.015 * gy)
    return label if abs(yf - gy) <= tol else f"{yf:.2f}y"


# ---------------- notional ----------------
def parse_notional(x):
    """Return (amount_in_currency_units, capped_flag).

    Handles 150000000, '150,000,000', '250MM+', '1.5BN', '750k'.
    SDR caps large notionals and appends '+' - keep that flag.
    """
    if isinstance(x, (int, float)) and not isinstance(x, bool):
        return (float(x), False) if np.isfinite(x) else (np.nan, False)
    s = str(x).strip().upper().replace(",", "").replace("€", "").replace("$", "").replace("£", "")
    if not s:
        return np.nan, False
    capped = s.endswith("+")
    s = s.rstrip("+").strip()
    mult = 1.0
    for suf, m in (("BN", 1e9), ("MM", 1e6), ("B", 1e9), ("M", 1e6), ("K", 1e3)):
        if s.endswith(suf):
            mult, s = m, s[: -len(suf)]
            break
    try:
        return float(s) * mult, capped
    except ValueError:
        return np.nan, capped


# ---------------- dv01 ----------------
def approx_dv01(notional, rate_pct, tenor_years, fwd_years=0.0):
    """Par-swap DV01 approximation: notional x annuity x 1bp.

    Annuity = (1 - (1+r)^-T)/r with the traded fixed rate as a flat discount
    proxy; forward-starting swaps get discounted by (1+r)^-t_fwd. Good to a
    few % of a proper curve-based DV01 - fine for ranking trade sizes.
    """
    if not (np.isfinite(notional) and np.isfinite(tenor_years)) or tenor_years <= 0:
        return np.nan
    r = (rate_pct or 0.0) / 100.0
    if not np.isfinite(r):
        r = 0.0
    ann = tenor_years if abs(r) < 1e-6 else (1.0 - (1.0 + r) ** -tenor_years) / r
    disc = (1.0 + r) ** -max(fwd_years, 0.0) if r > -1 else 1.0
    return notional * ann * disc * 1e-4


# ---------------- column normalisation ----------------
# SDR column headers differ between the SDR <GO> tabs, CSV exports and feeds.
# Map whatever arrives onto canonical names; first synonym found wins.
_COL_SYNONYMS = {
    "time":      ["trade time", "execution timestamp", "execution time", "exec time",
                  "timestamp", "time"],
    "effective": ["effective date", "effective", "start date", "eff date", "eff"],
    "maturity":  ["expiration date", "maturity date", "end date", "maturity",
                  "expiration", "expiry", "mat date"],
    "currency":  ["currency", "curr", "ccy", "notional currency", "notional currency 1"],
    "rate":      ["rate", "fixed rate", "fixed rate 1", "price", "strike", "coupon"],
    "notional":  ["notional", "notional amount", "notional amount 1", "notional 1", "amount"],
    "index":     ["index", "underlying", "floating rate index", "leg 2 index",
                  "reference rate", "underlier id", "und"],
    "platform":  ["platform", "sef", "exec venue", "execution venue", "venue",
                  "dissemination venue", "source", "dissem"],
    "product":   ["product", "type", "taxonomy", "asset class", "contract type",
                  "instrument"],
}

# Vanilla tab = fixed-float IRS (incl. ESTR OIS). Everything else out.
_PRODUCT_EXCLUDE = ("SWAPTION", "CAP", "FLOOR", "FRA", "XCCY", "CROSS",
                    "BASIS", "INFLATION", "ZC", "EXOTIC", "CDS")


def normalize_columns(raw):
    """Rename incoming columns to canonical names; leave unknown columns as-is."""
    df = raw.copy()
    lower = {str(c).strip().lower(): c for c in df.columns}
    ren = {}
    for canon, syns in _COL_SYNONYMS.items():
        for s in syns:
            if s in lower and lower[s] not in ren:
                ren[lower[s]] = canon
                break
    return df.rename(columns=ren)


# ---------------- pipeline ----------------
def build_table(raw, ccy="EUR", exclude_platforms=("TWSF", "TREU", "BBSF"),
                now=None):
    """raw trades DataFrame -> display table.

    Steps: normalise columns -> vanilla product filter -> currency filter ->
    platform exclusion -> parse American-format dates -> tenor / DV01 ->
    sort by time (newest first) -> tag same-timestamp trades as related
    packages (P1, P2, ...).
    """
    df = normalize_columns(raw)
    required = ["time", "effective", "maturity", "currency", "rate", "notional"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"SDR data is missing columns {missing}; got {list(raw.columns)}")

    for opt in ("index", "platform", "product"):
        if opt not in df.columns:
            df[opt] = ""

    # product filter (the Vanilla tab is already vanilla-only; this guards feeds
    # that deliver the whole Rates asset class)
    prod = df["product"].astype(str).str.upper()
    df = df[~prod.str.contains("|".join(_PRODUCT_EXCLUDE), na=False)]

    # currency + platform filters
    df = df[df["currency"].astype(str).str.strip().str.upper() == ccy.upper()]
    excl = {p.strip().upper() for p in exclude_platforms}
    df = df[~df["platform"].astype(str).str.strip().str.upper().isin(excl)]
    if df.empty:
        return pd.DataFrame(columns=["tenor", "rate", "notional", "dv01",
                                     "index", "time", "platform", "related"])

    # American-format dates (MM/DD/YYYY), tolerant of ISO too
    for c in ("effective", "maturity"):
        df[c] = pd.to_datetime(df[c], errors="coerce", dayfirst=False)
    df["time"] = pd.to_datetime(df["time"], errors="coerce", dayfirst=False)
    df = df.dropna(subset=["time", "effective", "maturity"])

    df["rate"] = pd.to_numeric(df["rate"], errors="coerce")
    parsed = df["notional"].map(parse_notional)
    df["notional_val"] = [p[0] for p in parsed]
    df["capped"] = [p[1] for p in parsed]

    now = now or pd.Timestamp.now()
    df["yf"] = [year_frac(e, m) for e, m in zip(df["effective"], df["maturity"])]
    df["fwd_years"] = ((df["effective"] - now).dt.days / 365.25).clip(lower=0.0)
    df["tenor"] = df["yf"].map(snap_tenor)
    df["dv01"] = [approx_dv01(n, r, t, f) for n, r, t, f in
                  zip(df["notional_val"], df["rate"], df["yf"], df["fwd_years"])]

    df = df.sort_values("time", ascending=False, kind="mergesort")

    # trades sharing an exact execution timestamp are related (package trades)
    sizes = df.groupby("time")["time"].transform("size")
    pkg_times = df.loc[sizes > 1, "time"].drop_duplicates().sort_values(ascending=False)
    pkg_id = {t: f"P{i + 1}" for i, t in enumerate(pkg_times)}
    df["related"] = df["time"].map(pkg_id).fillna("")

    return df[["tenor", "rate", "notional_val", "capped", "dv01", "index",
               "time", "platform", "related"]].rename(columns={"notional_val": "notional"})


def format_table(df):
    """Numeric table -> display strings for the grid."""
    out = pd.DataFrame(index=df.index)
    out["tenor"] = df["tenor"]
    out["rate"] = df["rate"].map(lambda r: f"{r:.4f}" if pd.notna(r) else "")
    out["notional"] = [
        (f"{n / 1e6:,.0f}mm" if n >= 1e6 else f"{n:,.0f}") + ("+" if c else "")
        if pd.notna(n) else ""
        for n, c in zip(df["notional"], df["capped"])
    ]
    out["dv01"] = df["dv01"].map(lambda d: f"{d:,.0f}" if pd.notna(d) else "")
    out["index"] = df["index"]
    same_day = df["time"].dt.normalize().nunique() <= 1
    fmt = "%H:%M:%S" if same_day else "%m/%d %H:%M:%S"
    out["time"] = df["time"].dt.strftime(fmt)
    out["platform"] = df["platform"]
    out["related"] = df["related"]
    return out.reset_index(drop=True)


In [ ]:
# ---------------- data layer: one function per source ----------------
# Every fetcher returns a *raw* DataFrame; normalisation happens downstream,
# so column names only need to be recognisable, not exact.

def fetch_via_bql():
    """SDR dissemination data via BQL (entitlement-dependent)."""
    if bq is None:
        raise RuntimeError("bql not importable - run inside BQuant")
    if not SDR_BQL_QUERY.strip():
        raise RuntimeError(
            "SDR_BQL_QUERY is empty. SDR dissemination data is an "
            "entitlement-dependent BQL dataset - paste your account's query "
            "string into the config cell, or use the CSV export source.")
    resp = bq.execute(SDR_BQL_QUERY)
    frames = [r.df().reset_index() for r in resp]
    df = pd.concat(frames, axis=1)
    return df.loc[:, ~df.columns.duplicated()]


def fetch_via_csv():
    """Re-read an SDR <GO> export each refresh (tick-by-re-export)."""
    import os
    if not os.path.exists(CSV_PATH):
        raise RuntimeError(
            f"'{CSV_PATH}' not found. In SDR <GO> (Rates / Vanilla tab) use "
            "Actions > Export, save the file next to this notebook.")
    return pd.read_csv(CSV_PATH)


def fetch_demo():
    """Synthetic prints so the pipeline/UI can be exercised anywhere."""
    rng = np.random.default_rng()
    now = pd.Timestamp.now().floor("s")
    curve = {"2y": 2.05, "3y": 2.10, "5y": 2.22, "7y": 2.35, "10y": 2.51,
             "15y": 2.65, "20y": 2.69, "30y": 2.60}
    platforms = ["DWSF", "TPSF", "ICAP", "CMSF", "OFF", "TWSF", "TREU", "BBSF"]
    rows, t = [], now
    for _ in range(40):
        t = t - pd.Timedelta(seconds=int(rng.integers(20, 400)))
        legs = [rng.choice(list(curve))]
        if rng.random() < 0.25:                       # package: 2-3 legs, same stamp
            legs += list(rng.choice(list(curve), size=int(rng.integers(1, 3))))
        # one venue / one currency per print - package legs stay together
        ccy = rng.choice(["EUR"] * 8 + ["USD", "GBP"])
        venue = rng.choice(platforms)
        for tenor in legs:
            yrs = int(tenor[:-1])
            eff = (now + pd.Timedelta(days=2)).normalize()
            mat = eff + pd.Timedelta(days=round(yrs * 365.25) + int(rng.integers(0, 4)))
            notional = float(rng.choice([25, 50, 75, 100, 150, 250])) * 1e6
            capped = notional >= 250e6
            rows.append({
                "Trade Time": t.strftime("%m/%d/%Y %H:%M:%S"),
                "Effective Date": eff.strftime("%m/%d/%Y"),
                "Expiration Date": mat.strftime("%m/%d/%Y"),
                "Curr": ccy,
                "Fixed Rate": round(curve[tenor] + rng.normal(0, 0.02)
                                    + (1.4 if ccy == "USD" else 0), 4),
                "Notional Amount 1": f"{notional:,.0f}" + ("+" if capped else ""),
                "Index": rng.choice(["ESTR"] * 7 + ["EURIBOR 6M"] * 3) if ccy == "EUR"
                         else ("SOFR" if ccy == "USD" else "SONIA"),
                "SEF": venue,
                "Product": "IRSwap:FixedFloat",
            })
    return pd.DataFrame(rows)


FETCHERS = {
    "SDR feed (BQL)": fetch_via_bql,
    "CSV export":     fetch_via_csv,
    "Demo data":      fetch_demo,
}


In [ ]:
# ---------------- UI (last cell - the app) ----------------
src_dd   = widgets.Dropdown(description="Source", options=list(FETCHERS),
                            value="Demo data", layout={"width": "220px"},
                            style={"description_width": "initial"})
ccy_dd   = widgets.Dropdown(description="Ccy", options=CCY_OPTIONS,
                            value=CCY_DEFAULT, layout={"width": "140px"},
                            style={"description_width": "initial"})
auto_tgl = widgets.ToggleButton(description="Auto", value=True, icon="clock-o",
                                tooltip="Auto-refresh", layout={"width": "80px"})
int_txt  = widgets.BoundedIntText(description="every (s)", value=REFRESH_SECONDS,
                                  min=5, max=600, layout={"width": "140px"},
                                  style={"description_width": "initial"})
btn      = widgets.Button(description="Refresh", button_style="primary",
                          icon="refresh", layout={"width": "110px"})
spinner  = widgets.HTML('<i class="fa fa-spinner fa-spin"></i>',
                        layout={"visibility": "hidden"})
status   = widgets.HTML("")
summary  = widgets.HTML("")

_EMPTY = pd.DataFrame(columns=["tenor", "rate", "notional", "dv01",
                               "index", "time", "platform", "related"])
if HAS_GRID:
    grid = DataGrid(_EMPTY, layout={"height": "520px"}, base_column_size=95,
                    column_widths={"index": 110, "time": 130, "notional": 110})
else:
    grid = widgets.HTML("<i>ipydatagrid not installed - using HTML table</i>")

_timer = None

def _render(disp):
    if HAS_GRID:
        grid.data = disp
    else:
        grid.value = disp.to_html(index=False, border=0)

def run(_=None):
    global _timer
    if _timer is not None:
        _timer.cancel()
        _timer = None
    spinner.layout.visibility = "visible"
    status.value = '<span style="color:#888">loading…</span>'
    try:
        raw = FETCHERS[src_dd.value]()
        table = build_table(raw, ccy=ccy_dd.value,
                            exclude_platforms=EXCLUDE_PLATFORMS).head(MAX_ROWS)
        _render(format_table(table))
        n_pkg = table.loc[table["related"] != "", "related"].nunique()
        tot_dv01 = np.nansum(table["dv01"].to_numpy(dtype=float))
        tot_ntl  = np.nansum(table["notional"].to_numpy(dtype=float))
        stamp = f"{datetime.now(APP_TZ):%H:%M:%S}" if APP_TZ else f"{datetime.now():%H:%M:%S}"
        demo = ' · <b style="color:#eda100">DEMO DATA</b>' if src_dd.value == "Demo data" else ""
        summary.value = (
            f'<div style="font:12px monospace;padding:4px 2px">'
            f'<b>{len(table)}</b> {ccy_dd.value} vanilla prints · '
            f'Σnotional <b>{tot_ntl/1e9:,.2f}bn</b> · '
            f'ΣDV01 <b>{tot_dv01/1e3:,.0f}k</b> · '
            f'<b>{n_pkg}</b> packages · excl {"/".join(EXCLUDE_PLATFORMS)}'
            f'{demo}</div>')
        status.value = f'<span style="color:#888">ok · {stamp}</span>'
    except Exception as e:
        status.value = f'<span style="color:#d62728">Error: {e}</span>'
    finally:
        spinner.layout.visibility = "hidden"
        if auto_tgl.value:
            _timer = threading.Timer(max(int_txt.value, 5), run)
            _timer.daemon = True
            _timer.start()

btn.on_click(run)
src_dd.observe(lambda ch: run() if ch["name"] == "value" else None, names="value")
ccy_dd.observe(lambda ch: run() if ch["name"] == "value" else None, names="value")
auto_tgl.observe(lambda ch: run() if ch["name"] == "value" and ch["new"] else None,
                 names="value")

app = widgets.VBox([
    widgets.HBox([src_dd, ccy_dd, auto_tgl, int_txt, btn, spinner, status]),
    summary, grid,
])
display(app)
run()
